# Manual message passing in NumPy

One round of message passing implemented by hand, to understand the mechanism before reaching for PyTorch Geometric. Each node updates its representation from its neighbors in three steps: gather, aggregate, update.

## The toy graph

A tiny directed transaction graph: five transactions, nodes 0 to 4. An edge `u -> v` means transaction `v` spends an output of `u`, so `v` receives from `u`. Node 2 has two parents (0 and 1), a many-to-one consolidation, then fans out to 3 and 4. Two features per node keep the math hand-checkable; treat them as stand-ins for the real Elliptic features.

In [1]:
import numpy as np

X = np.array([
    [1.0, 0.0],  # node 0
    [0.0, 1.0],  # node 1
    [1.0, 1.0],  # node 2
    [2.0, 0.0],  # node 3
    [0.0, 2.0],  # node 4
])

# Connectivity as an edge list, PyG style: row 0 holds sources, row 1
# holds targets. Column k is one edge, source -> target.
edge_index = np.array([
    [0, 1, 2, 2],  # sources
    [2, 2, 3, 4],  # targets
])

num_nodes = X.shape[0]
in_dim = X.shape[1]

## Aggregating from in-neighbors

In AML the signal is where the money came from, so each node aggregates from its sources, its in-neighbors. The edges are:

- `0 -> 2`
- `1 -> 2`
- `2 -> 3`
- `2 -> 4`

So node 2 receives from 0 and 1, and sends to 3 and 4.

## Worked example: node 2

Before writing a general function, the three steps on a single node. Node 2's in-neighbors are 0 and 1.

In [2]:
# gather: collect the in-neighbor feature vectors
messages = X[[0, 1]]

# aggregate: order-invariant mean across neighbors. This is what gives
# permutation invariance, the result does not depend on neighbor order.
neighbor_mean = messages.mean(axis=0)

# update: concatenate the node's own features with the aggregated
# neighbors. Concatenating instead of averaging lets the model later
# learn separate weights for 'my features' and 'my neighborhood'.
print('Node 2 hidden state:', np.concatenate([X[2], neighbor_mean]))

Node 2 hidden state: [1.  1.  0.5 0.5]


## One full round, for every node

Each node's hidden vector holds 4 values: 2 of its own plus 2 aggregated from neighbors. With 5 nodes that is a 5x4 matrix. To produce a 3-dimensional embedding, the weight matrix `W` maps 4 to 3, so it is 4x3, and the layer output is (5x4) @ (4x3) = 5x3.

In [3]:
def aggregate_mean(X, edge_index, num_nodes):
    # For each node, average the features of its in-neighbors (the
    # sources of edges pointing at it). One aggregated vector per node.
    sources, targets = edge_index[0], edge_index[1]
    agg = np.zeros_like(X)
    counts = np.zeros(num_nodes)

    # Walk every edge once. Each edge sends its source features to the
    # target and bumps the target's neighbor count.
    for src, tgt in zip(sources, targets):
        agg[tgt] += X[src]
        counts[tgt] += 1

    # Divide by the count to get the mean. Nodes with no in-neighbors
    # keep a zero vector, so guard against dividing by zero.
    for node in range(num_nodes):
        if counts[node] > 0:
            agg[node] /= counts[node]

    return agg


def message_passing_layer(X, edge_index, num_nodes, W):
    # One round: aggregate neighbors, concatenate each node with its own
    # features, then project through the linear layer.
    neighbor_agg = aggregate_mean(X, edge_index, num_nodes)
    concat = np.concatenate([X, neighbor_agg], axis=1)  # (num_nodes, 2 * in_dim)
    return concat @ W  # (num_nodes, out_dim)

In [4]:
out_dim = 3

# Fixed seed so the random weights are identical every run, which keeps
# the output reproducible while learning.
rng = np.random.default_rng(42)
W = rng.normal(size=(2 * in_dim, out_dim))  # (4, 3)

embeddings = message_passing_layer(X, edge_index, num_nodes, W)
print('aggregated neighbors:\n', aggregate_mean(X, edge_index, num_nodes))
print('embeddings shape:', embeddings.shape)
print('embeddings:\n', embeddings)

aggregated neighbors:
 [[0.  0. ]
 [0.  0. ]
 [0.5 0.5]
 [1.  1. ]
 [1.  1. ]]
embeddings shape: (5, 3)
embeddings:
 [[ 0.30471708 -1.03998411  0.7504512 ]
 [ 0.94056472 -1.95103519 -1.30217951]
 [ 0.88268003 -2.7094416  -0.17123292]
 [-0.11576936 -1.51681283  2.26189317]
 [ 1.15592591 -3.33891499 -1.84336824]]


## Sanity checks

- Node 2's aggregated row is `[0.5, 0.5]`, matching the hand calculation.
- Nodes 0 and 1 have no in-neighbors, so their aggregated row is `[0, 0]`.
- The output is `(5, 3)`. Stacking a second layer would feed this 5x3 matrix in as the next round's input, letting each node reach 2 hops out.

## Why the explicit loop

The per-edge loop is written for clarity: for each edge, send the source's features to the target. Real implementations vectorize this with a scatter-add, which is exactly what PyTorch Geometric does inside `SAGEConv`. These hand-written functions exist to understand the mechanism, not to ship in the pipeline; the project itself will use PyG's layer.